# Pitch parameter sweeps

This notebook is a launcher and results viewer; all expensive work runs in external, process-parallel scripts.

**Terminology:** `attune-uniform` names only Attune's observation-likelihood formulation. An **automatic winner** is the complete parameter tuple selected across all sampled tracks: observation mode, `max_jump_bins`, `switch_prob`, and `yin_trust`. The pYIN paper's transition constants are defined per frame. Since Attune's hop is half as long, the time-scaled constants are the literal paper baseline on Attune's frame grid; no separate unscaled-paper baseline is needed.

The first result directory retains the compact observation-formulation comparison. The second sweep fixes the winning `attune-uniform` observation mode and explores a denser 224-point grid over the other three parameters. Statistics combine all 48 stems, first averaging within each `(ensemble, instrument)` stratum so every represented family receives equal weight.

In [ ]:
from __future__ import annotations
import json
import os
import subprocess
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "app.py").is_file() and (path / "benchmarks").is_dir()
    )
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from benchmarks.modules.pitch.sweeps.PitchHMMParamSweep import (
    ATTUNE_OBSERVATION,
    DEFAULT_OUTPUT_DIR,
    PARAM_COLUMNS,
    ParameterAxes,
    SweepOptions,
    load_results,
    paper_parameters,
    parameter_grid,
    production_parameters,
    select_sample,
)
from benchmarks.modules.pitch.PitchNotebook import PitchNotebook

COMPACT_OUTPUT_DIR = DEFAULT_OUTPUT_DIR
ATTUNE_OUTPUT_DIR = DEFAULT_OUTPUT_DIR / "attune_grid"
CACHE_REFRESH_OUTPUT_DIR = DEFAULT_OUTPUT_DIR / "cache_refresh"
SWEEP_RUNNER = (
    REPO_ROOT
    / "benchmarks"
    / "notebooks"
    / "sweeps"
    / "scripts"
    / "pitch_hmm_param_sweep.py"
)
CACHE_RUNNER = (
    REPO_ROOT
    / "benchmarks"
    / "notebooks"
    / "sweeps"
    / "scripts"
    / "regenerate_pitch_caches.py"
)
WORKERS = max(1, (os.cpu_count() or 4) - 1)
FORCE_RECOMPUTE = False
FORCE_CACHE_REFRESH = False
OPTIONS = SweepOptions(seed=0, per_stratum=2, tune_per_stratum=1)
COMPACT_AXES = ParameterAxes()
ATTUNE_AXES = ParameterAxes(
    observation_mode=(ATTUNE_OBSERVATION,),
    max_jump_bins=(5, 9, 13, 17, 21, 25, 29, 33),
    switch_prob=(0.0025, 0.005, 0.01, 0.02),
    yin_trust=(0.25, 0.35, 0.45, 0.5, 0.55, 0.65, 0.75),
)


def csv_arg(values):
    return ",".join((str(value) for value in values))


def sweep_command(output_dir, axes, *, require_current_cache=False):
    command = [
        sys.executable,
        str(SWEEP_RUNNER),
        "--workers",
        str(WORKERS),
        "--output-dir",
        str(output_dir),
        "--split",
        OPTIONS.split,
        "--seed",
        str(OPTIONS.seed),
        "--per-stratum",
        str(OPTIONS.per_stratum),
        "--tune-per-stratum",
        str(OPTIONS.tune_per_stratum),
        "--observation-mode",
        csv_arg(axes.observation_mode),
        "--max-jump-bins",
        csv_arg(axes.max_jump_bins),
        "--switch-prob",
        csv_arg(axes.switch_prob),
        "--yin-trust",
        csv_arg(axes.yin_trust),
    ]
    if OPTIONS.max_strata is not None:
        command += ["--max-strata", str(OPTIONS.max_strata)]
    if OPTIONS.max_tracks is not None:
        command += ["--max-tracks", str(OPTIONS.max_tracks)]
    if require_current_cache:
        command.append("--require-current-cache")
    if FORCE_RECOMPUTE:
        command.append("--force")
    return command


print("repo:", REPO_ROOT)
print("workers:", WORKERS)
print("Attune-only variants:", len(parameter_grid(ATTUNE_AXES)))


In [ ]:
sample = select_sample(OPTIONS)
print(
    f"{len(sample)} stems across {sample.groupby(['ensemble', 'instrument']).ngroups} strata"
)
display(sample.groupby(["ensemble", "instrument"]).size().rename("tracks"))


## Compact observation-formulation comparison

This is the original compact sweep. `attune-uniform` is an observation mode, while the printed recommendation is the full parameter tuple that maximizes stratum-balanced Overall Accuracy across all 48 tracks.

In [ ]:
command = sweep_command(COMPACT_OUTPUT_DIR, COMPACT_AXES)
print(" ".join(command))
subprocess.run(command, cwd=REPO_ROOT, check=True)


In [ ]:
results = load_results(COMPACT_OUTPUT_DIR)
summary = results["summary"]
recommendation = results["recommendation"]
columns = [
    *PARAM_COLUMNS,
    "Tracks",
    "Errors",
    "Overall Accuracy",
    "Raw Pitch Accuracy",
    "Raw Chroma Accuracy",
    "Voicing Recall",
    "Voicing False Alarm",
    "voiced_fraction",
    "smoother_compute_time",
]
print("Automatic all-track winner (complete tuple):")
display(pd.DataFrame([{name: recommendation[name] for name in PARAM_COLUMNS}]))
print("Top all-track configurations:")
display(summary[columns].head(12))
named = {
    "Attune production": production_parameters(),
    "Literal paper (time-scaled to Attune hop)": paper_parameters(),
}
comparison_rows = []
for label, params in named.items():
    match = summary.copy()
    for name, value in params.items():
        match = match.loc[
            (
                np.isclose(match[name], value)
                if isinstance(value, float)
                else match[name].eq(value)
            )
        ]
    if not match.empty:
        comparison_rows.append(match.assign(configuration=label))
comparison = pd.concat(comparison_rows, ignore_index=True)
display(comparison[["configuration", *columns]])
comparison.set_index("configuration")["Overall Accuracy"].plot.bar(
    figsize=(9, 4), rot=15
)
plt.ylabel("Stratum-balanced Overall Accuracy")
plt.xlabel("")
plt.title("Attune observation model versus paper HMM")
plt.tight_layout()


## Regenerate raw pitch caches with the current YIN formulation

Run this cell before the dense sweep. It regenerates both raw and production-smoothed caches for the selected tracks in parallel. By default it skips caches already signed by the current Stage-1 detector; set `FORCE_CACHE_REFRESH = True` above to rebuild all selected caches.

In [ ]:
cache_command = [
    sys.executable,
    str(CACHE_RUNNER),
    "--workers",
    str(WORKERS),
    "--output-dir",
    str(CACHE_REFRESH_OUTPUT_DIR),
    "--split",
    OPTIONS.split,
    "--seed",
    str(OPTIONS.seed),
    "--per-stratum",
    str(OPTIONS.per_stratum),
    "--tune-per-stratum",
    str(OPTIONS.tune_per_stratum),
]
if OPTIONS.max_strata is not None:
    cache_command += ["--max-strata", str(OPTIONS.max_strata)]
if OPTIONS.max_tracks is not None:
    cache_command += ["--max-tracks", str(OPTIONS.max_tracks)]
if FORCE_CACHE_REFRESH:
    cache_command.append("--force")
print(" ".join(cache_command))
subprocess.run(cache_command, cwd=REPO_ROOT, check=True)
display(
    pd.read_csv(CACHE_REFRESH_OUTPUT_DIR / "cache_refresh.csv")["status"].value_counts()
)


## Dense Attune-uniform sweep

This fixes the observation mode to `attune-uniform` and evaluates 8 maximum jumps × 4 switch probabilities × 7 YIN-trust values = 224 configurations. The launcher refuses to run if any selected raw cache is not signed by the current Stage-1 detector.

In [ ]:
attune_grid = pd.DataFrame(parameter_grid(ATTUNE_AXES))
print(
    f"{len(sample)} stems × {len(attune_grid)} variants = {len(sample) * len(attune_grid):,} HMM evaluations"
)
display(attune_grid)


In [ ]:
command = sweep_command(ATTUNE_OUTPUT_DIR, ATTUNE_AXES, require_current_cache=True)
print(" ".join(command))
subprocess.run(command, cwd=REPO_ROOT, check=True)


In [ ]:
attune_results = load_results(ATTUNE_OUTPUT_DIR)
attune_summary = attune_results["summary"]
attune_recommendation = attune_results["recommendation"]
winner = {name: attune_recommendation[name] for name in PARAM_COLUMNS}
print("Dense-grid all-track winner (complete tuple):")
display(pd.DataFrame([winner]))
print("Top 20 all-track configurations:")
display(attune_summary[columns].head(20))
comparison_rows = []
for label, params in {
    "Dense all-track winner": winner,
    "Attune production": production_parameters(),
}.items():
    match = attune_summary.copy()
    for name, value in params.items():
        match = match.loc[
            (
                np.isclose(match[name], value)
                if isinstance(value, float)
                else match[name].eq(value)
            )
        ]
    comparison_rows.append(match.assign(configuration=label))
dense_comparison = pd.concat(comparison_rows, ignore_index=True)
print("All-track winner versus production:")
display(dense_comparison[["configuration", *columns]])
for parameter in ("max_jump_bins", "switch_prob", "yin_trust"):
    marginal = (
        attune_summary.groupby(parameter)["Overall Accuracy"]
        .agg(["max", "mean"])
        .sort_values("max", ascending=False)
    )
    print(parameter)
    display(marginal)


## Completed-audio pitch/voicing decoupling sweep

This cache-only experiment is the provenance for `attune_adhoc`. It compares the historical joint HMM with independent candidate tracking and the Praat-inspired global voicing grid. It never reruns pitch detection; Sarah launches it explicitly when the required Attune and Praat caches are ready.

In [ ]:
DECOUPLED_OUTPUT_DIR = DEFAULT_OUTPUT_DIR / "decoupled_voicing_preliminary"
decoupled_command = [
    sys.executable,
    "-m",
    "benchmarks.modules.pitch.sweeps.DecoupledVoicingSweep",
    "--output-dir",
    str(DECOUPLED_OUTPUT_DIR),
    "--seed",
    str(OPTIONS.seed),
]
print(" ".join(decoupled_command))
subprocess.run(decoupled_command, cwd=REPO_ROOT, check=True)


In [ ]:
decoupled_summary = pd.read_csv(DECOUPLED_OUTPUT_DIR / "summary_exact.csv")
decoupled_metadata = json.loads((DECOUPLED_OUTPUT_DIR / "metadata.json").read_text())
display(decoupled_summary)
display(pd.DataFrame(decoupled_metadata["praat_inspired_finalists"]).T)


## Attune real-time voicing sweeps

These diagnostics were moved out of the main comparison notebook. They re-score cached `attune_realtime` evidence and do not rerun pitch detection.

In [ ]:
SWEEP_BENCH = PitchNotebook(PitchNotebook.NotebookConfig())
preliminary_urmp = SWEEP_BENCH.run_preliminary_urmp()


In [ ]:
urmp_voicing_thresholds = SWEEP_BENCH.show_attune_realtime_voicing_thresholds(
    preliminary_urmp
)


In [ ]:
urmp_volume_floors = SWEEP_BENCH.show_attune_realtime_volume_floors(preliminary_urmp)
